In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load a chunk of data to keep things fast
df = pd.read_csv('paysim dataset.csv', nrows=200000)

# Show the first 5 rows to see the columns clearly
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'paysim dataset.csv'

In [ ]:
# Create a summary of Fraud vs Legitimate transactions
fraud_counts = df['isFraud'].value_counts()
print(f"Legit Transactions: {fraud_counts[0]}")
print(f"Fraud Transactions: {fraud_counts[1]}")

# Let's see which 'type' of transaction fraudsters prefer
fraud_types = df[df['isFraud'] == 1]['type'].value_counts()
print("\nFraud happens in these transaction types:") 
print(fraud_types)

In [ ]:
%pip install networkx
import networkx as nx

# 1. Take only the fraud cases we found
fraud_df = df[df['isFraud'] == 1]

# 2. Create a "Graph" object
G = nx.Graph()

# 3. Add connections: from nameOrig (Sender) to nameDest (Receiver)
for index, row in fraud_df.iterrows():
    G.add_edge(row['nameOrig'], row['nameDest'], amount=row['amount'])

# 4. Draw the map
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, k=0.5) # Spreads the dots out
nx.draw_networkx(G, pos, with_labels=False, node_size=30, node_color="red", alpha=0.6)
plt.title("The Fraud Network: Who is sending to whom?")
plt.show()

print(f"Nodes (Accounts): {G.number_of_nodes()}")
print(f"Edges (Transactions): {G.number_of_edges()}")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Step 1 — filter to fraud-relevant types and confirmed fraud only
fraud_df = df[
    (df['type'].isin(['TRANSFER', 'CASH_OUT'])) &
    (df['isFraud'] == 1)
]

print(f"Fraud transactions: {len(fraud_df)}")

# Step 2 — build the graph
G = nx.from_pandas_edgelist(
    fraud_df,
    source='nameOrig',
    target='nameDest',
    edge_attr='amount',
    create_using=nx.DiGraph
)

# Step 3 — colour nodes: red = fraud origin, orange = fraud dest, grey = other
fraud_orig = set(fraud_df['nameOrig'])
fraud_dest = set(fraud_df['nameDest'])

color_map = []
for node in G.nodes():
    if node in fraud_orig:
        color_map.append('#E24B4A')   # confirmed fraudster
    elif node in fraud_dest:
        color_map.append('#BA7517')   # mule/destination account
    else:
        color_map.append('#B4B2A9')   # clean account

# Step 4 — spring layout clusters tightly connected nodes together
pos = nx.spring_layout(G, k=0.5, seed=42)

plt.figure(figsize=(14, 10))
nx.draw_networkx(G, pos,
        node_color=color_map,
        node_size=60,
        edge_color='#888780',
        arrows=True,
        arrowsize=8,
        width=0.5,
        alpha=0.85)

plt.title(f"Fraud transaction network — {G.number_of_nodes()} accounts, {G.number_of_edges()} transactions")
plt.tight_layout()
plt.show()

# Step 5 — extract graph features for ML
degree_centrality = nx.degree_centrality(G)
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

fraud_df = fraud_df.copy()
fraud_df['orig_out_degree'] = fraud_df['nameOrig'].map(out_degree).fillna(0)
fraud_df['dest_in_degree']  = fraud_df['nameDest'].map(in_degree).fillna(0)
fraud_df['orig_centrality'] = fraud_df['nameOrig'].map(degree_centrality).fillna(0)

print(fraud_df[['nameOrig','nameDest','amount','orig_out_degree','dest_in_degree']].head(10))

In [ ]:
import networkx as nx

fraud_df = df[df['isFraud'] == 1].copy()

# CHANGE 1: DiGraph preserves direction of money flow
G = nx.DiGraph()
for _, row in fraud_df.iterrows():
    G.add_edge(row['nameOrig'], row['nameDest'], amount=row['amount'])

in_deg  = dict(G.in_degree())    # high = mule aggregator
out_deg = dict(G.out_degree())   # high = fraud ring leader

# CHANGE 2: use in_degree for destinations, out_degree for origins
fraud_df['orig_out_degree'] = fraud_df['nameOrig'].map(out_deg).fillna(0).astype(int)
fraud_df['dest_in_degree']  = fraud_df['nameDest'].map(in_deg).fillna(0).astype(int)

# CHANGE 3: threshold on in_degree now makes sense
mule_hubs = {n: d for n, d in in_deg.items() if d > 1}
print(f"Mule hub accounts (received from 2+ senders): {len(mule_hubs)}")
print(f"Top 10 mule hubs:")
print(sorted(mule_hubs.items(), key=lambda x: x[1], reverse=True)[:10])